In [15]:
import json
from pathlib import Path

import pandas as pd

_example_path = Path("example_data.json")
if not _example_path.exists():
    _example_path = Path("src/analysis/example_data.json")
with open(_example_path) as f:
    data = json.load(f)

df = pd.DataFrame(data["result"])
df

,regYear,tenant,aptStat,filingDate,legalRent,prefRent,paidRent,reasonsChange,leaseStart,leaseEnd,note
0,1984-I,Joseph Monaco,RS,NC,425.0,NaN,NaN,[IMPRVMNT],NaN,08/31/1984,NaN
1,1985,Joseph Monaco,RS,NC,442.0,NaN,NaN,[],NaN,08/31/1985,NaN
2,1986,Joseph Monaco,RS,NC,468.52,NaN,NaN,[LEAS/RNL],09/01/1985,08/31/1986,NaN
3,1987,Joseph Monaco,RS,NC,487.26,NaN,NaN,[LEAS/RNL],09/01/1986,08/31/1987,NaN
4,1988,Joseph Monaco,RS,10/17/1988,516.5,NaN,NaN,[LEAS/RNL],09/01/1987,08/31/1988,NaN
5,1989,Joseph Monaco,RS,07/07/1989,532.0,NaN,NaN,[LEAS/RNL],09/01/1988,08/31/1989,NaN
6,1990,Joseph Monaco,RS,06/28/1990,563.91,NaN,NaN,[LEAS/RNL],09/01/1989,08/31/1990,NaN
7,1991,Joseph Monaco,RS,07/05/1991,594.97,NaN,NaN,[LEAS/RNL],09/01/1990,08/31/1991,NaN
8,1992,Joseph Monaco,RS,04/16/1992,621.74,NaN,NaN,[LEAS/RNL],09/01/1991,08/31/1992,NaN
9,1993,Joseph Monaco,RS,06/11/1993,646.61,NaN,NaN,[LEAS/RNL],09/01/1992,08/31/1993,NaN


In [16]:
REFERENCE_TABLE = pd.DataFrame([
    {"regYear": "1985", "maxIncreasePct": 6.0, "guideline": "RGB"},
    {"regYear": "1986", "maxIncreasePct": 6.0, "guideline": "RGB"},
    {"regYear": "1987", "maxIncreasePct": 4.0, "guideline": "RGB"},
])

REFERENCE_TABLE

,regYear,maxIncreasePct,guideline
0,1985,6.0,RGB
1,1986,6.0,RGB
2,1987,4.0,RGB


In [17]:
def compare_two_rows(
    df: pd.DataFrame,
    year_earlier: str,
    year_later: str,
    ref: pd.DataFrame,
    rent_col: str = "legalRent",
) -> dict:
    """
    Compare two rows by regYear and apply formula using reference table.
    Returns dict with earlier row, later row, computed values, and reference check.
    """
    row_a = df[df["regYear"] == year_earlier].iloc[0]
    row_b = df[df["regYear"] == year_later].iloc[0]

    def _to_float(x):
        try:
            return float(x)
        except (TypeError, ValueError):
            return None
    rent_a = _to_float(row_a[rent_col])
    rent_b = _to_float(row_b[rent_col])

    # Formula: percent change in legal rent (skip if either rent is non-numeric, e.g. "EXEMPT")
    if rent_a is not None and rent_b is not None and rent_a > 0:
        pct_change = 100.0 * (rent_b - rent_a) / rent_a
    else:
        pct_change = None

    # Look up reference max for the later year
    ref_row = ref[ref["regYear"] == year_later]
    max_pct = ref_row["maxIncreasePct"].iloc[0] if len(ref_row) else None

    within_guideline = (
        pct_change is not None and max_pct is not None and pct_change <= max_pct
    )

    return {
        "year_earlier": year_earlier,
        "year_later": year_later,
        "rent_earlier": rent_a,
        "rent_later": rent_b,
        "pct_change": pct_change,
        "ref_max_increase_pct": max_pct,
        "within_guideline": within_guideline,
        "row_earlier": row_a,
        "row_later": row_b,
    }


# Example: compare 1985 vs 1986
result = compare_two_rows(df, "1985", "1986", REFERENCE_TABLE)
for k, v in result.items():
    if k not in ("row_earlier", "row_later"):
        print(f"{k}: {v}")
print("---")
print("row_earlier:", result["row_earlier"].to_dict())
print("row_later:", result["row_later"].to_dict())

year_earlier: 1985
year_later: 1986
rent_earlier: 442.0
rent_later: 468.52
pct_change: 5.999999999999996
ref_max_increase_pct: 6.0
within_guideline: True
---
row_earlier: {'regYear': '1985', 'tenant': 'Joseph Monaco', 'aptStat': 'RS', 'filingDate': 'NC', 'legalRent': 442.0, 'prefRent': nan, 'paidRent': nan, 'reasonsChange': [], 'leaseStart': nan, 'leaseEnd': '08/31/1985', 'note': nan}
row_later: {'regYear': '1986', 'tenant': 'Joseph Monaco', 'aptStat': 'RS', 'filingDate': 'NC', 'legalRent': 468.52, 'prefRent': nan, 'paidRent': nan, 'reasonsChange': ['LEAS/RNL'], 'leaseStart': '09/01/1985', 'leaseEnd': '08/31/1986', 'note': nan}


In [18]:
# sort by leading numeric part so values like "1984-I" and "1985" order correctly
def _year_sort_key(y):
    s = str(y).split("-")[0]
    return int(s) if s.isdigit() else 0

years = sorted(df["regYear"].astype(str).unique(), key=_year_sort_key)
comparisons = []
for i in range(len(years) - 1):
    y1, y2 = years[i], years[i + 1]
    comp = compare_two_rows(df, y1, y2, REFERENCE_TABLE)
    comparisons.append({
        "year_earlier": comp["year_earlier"],
        "year_later": comp["year_later"],
        "pct_change": comp["pct_change"],
        "within_guideline": comp["within_guideline"],
    })

pd.DataFrame(comparisons)

,year_earlier,year_later,pct_change,within_guideline
0,1984-I,1985,4.000000,True
1,1985,1986,6.000000,True
2,1986,1987,3.999829,True
3,1987,1988,6.000903,False
4,1988,1989,3.000968,False
5,1989,1990,5.998120,False
6,1990,1991,5.507971,False
7,1991,1992,4.499387,False
8,1992,1993,4.000064,False
9,1993,1994,3.000263,False
